# GPU Performance Engineering

This tutorial will explore how to evaluate and optimize the performance of GPU-accelerated codes.
The goal is to:
- Understand the expected optimal performance of your code.
- Identify how far the actual performance deviates from the optimal.
- Explore reasons for performance gaps and practical methods to bridge them.

### What This Tutorial Covers

- **GPU architecture fundamentals:** Learn how GPUs are designed and how they execute code.
- **Simplistic performance modeling:** Use basic models to predict performance characteristics.
- **Micro-benchmarks:** Isolate specific hardware effects and understand performance limits.
- **Performance engineering workflow:** Follow a structured process to optimize real-world applications.
- **Tools and techniques:** Use NVIDIA tools such as Nsight Systems and Nsight Compute, and NVTX markers for profiling.

### What This Tutorial Does NOT Cover

- **Algorithm engineering:** While algorithm choice and tuning are critical, they are outside the scope of this tutorial.

## What is Performance

Performance is generally considered as the amount of useful work done per unit time.
This can be captured as work done in a given time interval, or the time required to perform a fixed amount of work.

Different performance *metrics* help evaluate and categorize performance:

### 1. Time-Based Metrics

**Time to Solution (TTS)**: Measures total execution time for the full application, usually including setup & initialization, shutdown, *the actual meaningful work* and anything else that might be going on the in program to examine.

Example:
```bash
time ./my-app
```

Pros:
* Very easy to set up
* Captures all effects at once

Cons:
* Captures all effects at once
* Comparing different applications (or their parameterization) *in a meaningful way* is challenging
* Assessing potential performance improvements is almost impossible

Potential improvements: more narrow scope and normalization over parameterization.
Tools can help with the former, but often struggle with the latter.

A manual approach facilitating both is adding custom timers or timed regions to the code.Options include `std::chrono`, MPI and OpenMP timers.
This course will mostly work with the former.

Example (in a C++ program):

```cpp
#include <chrono>

// ...
int main(int argc, char *argv[]) {
    // work not timed

    auto start = std::chrono::steady_clock::now();

    // work to be timed

    auto end = std::chrono::steady_clock::now();

    const std::chrono::duration<double> elapsedSeconds = end - start;

    std::cout << "Time taken: " << 1e3 * elapsedSeconds.count() << " ms\n";

    // other work not timed
}
```

Pros:
* Very flexible
* Allows capturing of more fine-granular effects
* Can already pinpoint hot spots of the code

Cons:
* More manual work
* Requires additional interpretation to relate timings to performance effects

### 2. Application-Specific Metrics


Depending on the application at hand, normalization of timings can be helpful to allow for better comparability and yielding of further insights.

This course will mainly build on two variants:
* **Iterations per Second (It/s)**: Normalizes measured times across a flexible iteration count.
* **Mega Lattice Site Updates per Second (MLUPS)**: Expresses throughput in terms of updates per grid cell.

Pros:
* Allows comparing performance of different problem/ work sizes
* Can be related to other metrics (see below) more easily

Cons:
* Limited insight into optimization potential

### 3. Hardware-Centric Metrics

Once timings for isolated code regions are available, their performance can be assessed in terms of different metrics.
Generally speaking, we are interested in two quantities:
* **Latency**: how long until a certain operation is completed, e.g.
  * The time required to access a single element of data in main memory.
  * The time required to compute a single square root in a given precision.
* **Throughput**: what is the rate of operations averaged over the timed region, e.g.
  * The *computational throughput* in floating point operations per second (FLOPS or FLOP/s), or in integer operations per second (IOPS or IOP/s).
  * The *instruction throughput* in instructions per second (IPS).
  * The *data throughput* and bandwidth in bytes per second (B/s).

Example (from [util.h](../src/util.h]), assuming some abstract quantity of work as *cells*):

```cpp
template<typename tpe>
void printStats(const std::chrono::duration<double> elapsedSeconds, size_t nIt, size_t nCells, char* tpeName, size_t numBytesPerCell, size_t numFlopsPerCell) {
    std::cout << "  #cells / #it:  " << nCells << " / " << nIt << "\n";
    std::cout << "  type:          " << tpeName << "\n";
    std::cout << "  elapsed time:  " << 1e3 * elapsedSeconds.count() << " ms\n";
    std::cout << "  per iteration: " << 1e3 * elapsedSeconds.count() / nIt << " ms\n";
    std::cout << "  MLUP/s:        " << 1e-6 * nCells * nIt / elapsedSeconds.count() << "\n";
    std::cout << "  bandwidth:     " << 1e-9 * numBytesPerCell * nCells * nIt / elapsedSeconds.count() << " GB/s\n";
    std::cout << "  compute:       " << 1e-9 * numFlopsPerCell * nCells * nIt / elapsedSeconds.count() << " GFLOP/s\n";
}
```

Pros:
* Allows comparison against theoretical performance limits (hardware capabilities)
* Enables performance *prediction* for different hardware platforms

Cons:
* Requires profiling tools or assumptions about workload (bytes transferred, etc.)
* Relating profiling results and application can be challenging

### Alternative Considerations

Performance is also often expressed in terms of *work per energy consumed*, especially in the field of power-efficient or 'green' computing.

And, of course, arbitrary other metrics are also conceivable, e.g. *investment cost over energy to solution* in *USD per Joule*, but frequently of limited use in practice.

## Next Step

We start by investigating some of this metrics at the example of a straight-forward test case.
Head over to the [Stencil Test Case](./stencil-test-case.ipynb) notebook to get started.